# 02 — GARCH(1,1) (weekly silver volatility)

GARCH(1,1) is the classic parametric benchmark for conditional volatility. Unlike
HAR-RV — which regresses *realised* volatility on its own lags — GARCH models the
volatility of the **return series itself**, so it provides an econometric reference
point of a different kind.

The weekly return series and the RV target both come from `volatility_weekly.csv`,
built in `00_features.ipynb` — run that notebook first.


## Setup


In [1]:
import sys, os
sys.path.append(os.path.abspath('../../src'))

import numpy as np
import pandas as pd
from arch import arch_model
from vol_utils import vol_evaluate, vol_period_metrics
from eval_utils import PERIODS
import warnings; warnings.filterwarnings('ignore')

frame = pd.read_csv('../../data/processed/volatility_weekly.csv',
                    parse_dates=['Date']).set_index('Date')
test_df = frame[frame['split'] == 'test']

y_test     = test_df['target'].values
prev_test  = test_df['rv_w_lag1'].values
weekly_ret = frame['silver_ret']               # weekly silver log-returns, full sample
print(f'test weeks: {len(test_df)}')


test weeks: 175


## 1. Walk-forward GARCH(1,1)

The model assumes

$$r_t = \mu + \varepsilon_t, \qquad \varepsilon_t = \sigma_t z_t, \qquad
\sigma_t^2 = \omega + \alpha\,\varepsilon_{t-1}^2 + \beta\,\sigma_{t-1}^2.$$

The one-step-ahead conditional volatility $\hat\sigma_t$ is used as the prediction for
$\text{RV}_t$. To avoid look-ahead the model is **re-fit walking forward week by
week**: at each test week the GARCH is estimated only on weekly returns strictly prior
to that week, then forecast one step ahead. Returns are scaled by 100 during fitting
for numerical stability, and the forecast is scaled back afterwards.


In [2]:
garch_preds = []
for d in test_df.index:
    hist = weekly_ret.loc[:d].iloc[:-1]                            # returns strictly before week d
    m   = arch_model(hist * 100, mean='Constant', vol='GARCH', p=1, q=1, dist='normal')
    res = m.fit(disp='off', show_warning=False)
    fc  = res.forecast(horizon=1, reindex=False)
    garch_preds.append(np.sqrt(fc.variance.values[-1, 0]) / 100)   # back to return scale
garch_preds = np.array(garch_preds)
print(f'fitted {len(garch_preds)} walk-forward GARCH models')


fitted 175 walk-forward GARCH models


## 2. Evaluation


In [3]:
results = [vol_evaluate('GARCH(1,1)', y_test, garch_preds, prev_test)]


GARCH(1,1)                      RMSE=0.03296  MAE=0.01824  R2=+0.249  DCA=0.674


## 3. Sub-period breakdown

RMSE and DCA split by calendar year, using the shared `PERIODS` definition.


In [4]:
period_garch = vol_period_metrics(y_test, garch_preds, prev_test, test_df.index, PERIODS)
period_garch.to_csv('../../data/processed/period_garch_volatility.csv')
period_garch.round(4)


,n,RMSE,DCA
Period,,,
2023 (choppy),52,0.0162,0.6731
2024 (bull start),52,0.0145,0.7885
2025 (bull run),52,0.0251,0.6346
2026 (YTD),19,0.0836,0.4737
── Full test ──,175,0.0330,0.6743


## 4. Save outputs

- `metrics_garch_volatility.csv` — GARCH(1,1) headline metrics
- `pred_garch_volatility.csv` — test-set predictions, consumed by `evaluation.ipynb`


In [5]:
pd.DataFrame(results).to_csv('../../data/processed/metrics_garch_volatility.csv', index=False)

pred_garch = pd.DataFrame({'actual': y_test, 'prev': prev_test, 'garch': garch_preds},
                          index=test_df.index)
pred_garch.to_csv('../../data/processed/pred_garch_volatility.csv', index_label='Date')
print('Saved metrics_garch_volatility.csv + pred_garch_volatility.csv')
pd.DataFrame(results).round(5)


Saved metrics_garch_volatility.csv + pred_garch_volatility.csv


,model,rmse,mae,r2,dca
0,"GARCH(1,1)",0.03296,0.01824,0.24899,0.67429
